In [ ]:
# Load Target and predictions

# Deterministic models
det_models = {"UNET": "/mimer/NOBACKUP/groups/mlhighres/users/erifh/neural-lam/output/SAMPLES-subset-eval-test-unet-6x128-01_18_15-5264"}

# Probabilistic models
prob_models = {
    "SI": "/mimer/NOBACKUP/groups/mlhighres/users/erifh/neural-lam/output/SAMPLES-subset-eval-test-SI-6x128-01_18_15-8562",
    "EDM": "/mimer/NOBACKUP/groups/mlhighres/users/erifh/neural-lam/output/SAMPLES-subset-eval-test-diffusion-6x128-01_18_15-3658",
    "CorrDiff": "/mimer/NOBACKUP/groups/mlhighres/users/erifh/neural-lam/output/SAMPLES-subset-eval-test-CorrDiff-6x128-01_18_15-1651"
}

# Load data
import numpy as np
import torch

device = torch.device("cpu")

def load_prob_data(path):
    ens_mean = torch.load(f"{path}/example_ens_mean_1.pt", map_location=device).numpy()
    ens_members = torch.load(f"{path}/example_ens_members_1.pt", map_location=device).numpy()
    ens_std = torch.load(f"{path}/example_ens_std_1.pt", map_location=device).numpy()
    target = torch.load(f"{path}/example_target_1.pt", map_location=device).numpy()
    return ens_mean, ens_members, ens_std, target

def load_det_data(path):
    prediction = torch.load(f"{path}/pred_1.pt", map_location=device).numpy()
    target = torch.load(f"{path}/example_target_1.pt", map_location=device).numpy()
    return prediction, target


In [ ]:
import os

data = {}

# deterministic
for name, path in det_models.items():
    entry = {"type": "deterministic", "path": path}
    try:
        pred, target = load_det_data(path)
        entry["prediction"] = pred
        entry["target"] = target
    except Exception as e:
        print(f"Warning: failed to load deterministic model {name} from {path}: {e}")
        entry["error"] = str(e)
    data[name] = entry

# probabilistic
for name, path in prob_models.items():
    entry = {"type": "probabilistic", "path": path}
    try:
        ens_mean, ens_members, ens_std, target = load_prob_data(path)
        entry["ens_mean"] = ens_mean
        entry["ens_members"] = ens_members
        entry["ens_std"] = ens_std
        entry["target"] = target
    except Exception as e:
        print(f"Warning: failed to load probabilistic model {name} from {path}: {e}")
        entry["error"] = str(e)
    data[name] = entry

# quick summary
for name, entry in data.items():
    if "error" in entry:
        print(f"{name}: ERROR -> {entry['error']}")
    else:
        if entry["type"] == "deterministic":
            print(f"{name}: pred {entry['prediction'].shape}, target {entry['target'].shape}")
        else:
            print(f"{name}: ens_mean {entry['ens_mean'].shape}, members {entry['ens_members'].shape}, std {entry['ens_std'].shape}, target {entry['target'].shape}")

In [ ]:
# ...existing code...
# plotting cell (replace existing plotting block with this)
import numpy as np
import matplotlib.pyplot as plt

# choose variable/channel to plot (0 or 1)
VAR_IDX = 0

# set grid shape if known, otherwise None to auto-detect
GRID_SHAPE = (400, 550)  # or None

# whether to include std row
PLOT_STD = True

def infer_grid_shape(N):
    candidates = [(400, 550), (550, 400)]
    for h, w in candidates:
        if h * w == N:
            return (h, w)
    for h in range(int(np.sqrt(N)), 0, -1):
        if N % h == 0:
            return (h, N // h)
    raise ValueError(f"Cannot infer grid shape for N={N}")

def best_reshape(vec, gs):
    vec = np.asarray(vec).ravel()
    opts = []
    try:
        opts.append(vec.reshape(gs))                # C-order
    except Exception:
        pass
    try:
        opts.append(vec.reshape(gs, order='F'))     # Fortran-order
    except Exception:
        pass
    try:
        opts.append(vec.reshape((gs[1], gs[0])).T)  # swapped dims + transpose
    except Exception:
        pass
    if not opts:
        return vec.reshape(gs)  # last-resort
    # score by mean absolute gradient (spatial variability)
    scores = []
    for opt in opts:
        gx = np.abs(np.diff(opt, axis=0)).mean() if opt.shape[0] > 1 else 0.0
        gy = np.abs(np.diff(opt, axis=1)).mean() if opt.shape[1] > 1 else 0.0
        scores.append(gx + gy)
    return opts[int(np.argmax(scores))]

def to2d(a, var=0, grid_shape=None):
    a = np.asarray(a)
    # already spatial image
    if a.ndim == 2 and a.shape[0] > 4 and a.shape[1] > 4:
        return a
    # (C, H, W) -> channel
    if a.ndim == 3 and a.shape[0] <= 4:
        return a[var]
    # (H, W, C) -> channel
    if a.ndim == 3 and a.shape[-1] <= 4:
        return a[..., var]
    # (N_flat, C) common case, e.g. (220000, 2)
    if a.ndim == 2 and a.shape[1] <= 8:
        N, C = a.shape
        gs = grid_shape or infer_grid_shape(N)
        # reshape to (H, W, C) if possible then select var
        try:
            img3 = a.reshape(*gs, C)
            return img3[..., var]
        except Exception:
            # fallback: reshape single-channel vec with heuristics
            vec = a[:, var]
            return best_reshape(vec, gs)
    # (n_members, N_flat, C) -> take first member
    if a.ndim == 3 and a.shape[-1] <= 8 and a.shape[0] > 1:
        # treat as (n_members, N, C)
        member = a[0]
        return to2d(member, var=var, grid_shape=grid_shape)
    # 1D flat
    if a.ndim == 1:
        N = a.size
        gs = grid_shape or infer_grid_shape(N)
        return best_reshape(a, gs)
    raise ValueError(f"Unhandled array shape in to2d: {a.shape}")

# prepare data lists
models = list(data.keys())
n_models = len(models)

# find a target (first entry that contains 'target')
target_entry = next((e for e in data.values() if 'target' in e), None)
target_img = to2d(target_entry['target'], VAR_IDX, GRID_SHAPE) if target_entry is not None else None

mean_imgs = []
member_imgs = []
std_imgs = []

for name in models:
    entry = data[name]
    if entry.get("type") == "probabilistic":
        mean = entry["ens_mean"]
        members = np.asarray(entry["ens_members"])
        std = entry["ens_std"]
        if members.ndim == 3:
            member = members[0]
        elif members.ndim == 2:
            member = members
        else:
            member = members.reshape(-1, members.shape[-1])[:, :mean.shape[1]]
    else:
        mean = entry.get("prediction")
        member = entry.get("prediction")
        std = np.zeros_like(mean)

    mean_imgs.append(to2d(mean, VAR_IDX, GRID_SHAPE))
    member_imgs.append(to2d(member, VAR_IDX, GRID_SHAPE))
    std_imgs.append(to2d(std, VAR_IDX, GRID_SHAPE))

# compute vmin/vmax over all images we'll display (include target if present)
all_imgs = []
if target_img is not None:
    all_imgs.append(target_img)
all_imgs += mean_imgs + member_imgs
if PLOT_STD:
    all_imgs += std_imgs

vmin = float(min(img.min() for img in all_imgs))
vmax = float(max(img.max() for img in all_imgs))

# rows: mean, member, optional std
rows = 2 + (1 if PLOT_STD else 0)
cols = 1 + n_models  # first column reserved for target

fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows), squeeze=False)
cmap = "plasma"
FONTSIZE = 20

# plot target in left column (only in first row)
if target_img is not None:
    axes[0, 0].set_title("Target", fontsize=FONTSIZE + 2, fontweight="bold")
    im0 = axes[0, 0].imshow(target_img, cmap=cmap, origin="lower", vmin=vmin, vmax=vmax)
    axes[0, 0].axis("off")
else:
    axes[0, 0].axis("off")
# hide other left-column rows
for r in range(1, rows):
    axes[r, 0].axis("off")

# plot each model in columns 1..n
for j, name in enumerate(models):
    col = j + 1
    axes[0, col].set_title(f"{name} Mean", fontsize=FONTSIZE + 2, fontweight="bold")
    im_ref = axes[0, col].imshow(mean_imgs[j], cmap=cmap, origin="lower", vmin=vmin, vmax=vmax)
    axes[0, col].axis("off")

    axes[1, col].set_title("Ensemble Member", fontsize=FONTSIZE, fontweight="bold")
    axes[1, col].imshow(member_imgs[j], cmap=cmap, origin="lower", vmin=vmin, vmax=vmax)
    axes[1, col].axis("off")

    if PLOT_STD:
        axes[2, col].set_title("Std", fontsize=FONTSIZE, fontweight="bold")
        last_std_mappable = axes[2, col].imshow(std_imgs[j], cmap=cmap, origin="lower", vmin=0, vmax=1)
        axes[2, col].axis("off")

# colorbar: use reference image (prefer target if present else first model)
ref_im = im0 if (target_entry is not None) else im_ref
fig.subplots_adjust(bottom=0., hspace=0.2, wspace=0.02)
# fig.colorbar(ref_im, ax=axes[0,0], orientation="vertical", fraction=0.06, pad=0.05)

cb_axes_main = [axes[r, c] for r in range(0, 2) for c in range(0, cols)]
cbar_main = fig.colorbar(ref_im, ax=cb_axes_main, orientation="vertical", fraction=0.02, pad=0.01, aspect=14)
cbar_main.ax.tick_params(labelsize=12)

# plt.tight_layout()

if PLOT_STD:
    cb_axes_std = [axes[2, c] for c in range(0, cols)]
    cbar_std = fig.colorbar(last_std_mappable, ax=cb_axes_std, orientation="vertical", fraction=0.02, pad=0.01, aspect=7)
    cbar_std.ax.tick_params(labelsize=12)

out_pdf = "/mimer/NOBACKUP/groups/mlhighres/users/erifh/neural-lam/plots/qualitative_comparison_plot.pdf"
fig.savefig(out_pdf, format="pdf", bbox_inches="tight")
print(f"Saved figure to {out_pdf}")

plt.show()
# ...existing code...

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy import fftpack
VAR_IDX = 0  # variable/channel index to analyze

plots_dir = "/mimer/NOBACKUP/groups/mlhighres/users/erifh/neural-lam/plots/spectra"
os.makedirs(plots_dir, exist_ok=True)

# if to2d exists in the notebook use it, otherwise define a small fallback
try:
    to2d  # noqa: F821
except NameError:
    def infer_grid_shape(N):
        for h, w in [(400, 550), (550, 400)]:
            if h * w == N:
                return (h, w)
        for h in range(int(np.sqrt(N)), 0, -1):
            if N % h == 0:
                return (h, N // h)
        raise ValueError(f"Cannot infer grid shape for N={N}")

    def best_reshape(vec, gs):
        vec = np.asarray(vec).ravel()
        opts = []
        try:
            opts.append(vec.reshape(gs))
        except Exception:
            pass
        try:
            opts.append(vec.reshape(gs, order="F"))
        except Exception:
            pass
        try:
            opts.append(vec.reshape((gs[1], gs[0])).T)
        except Exception:
            pass
        if not opts:
            return vec.reshape(gs)
        scores = []
        for opt in opts:
            gx = np.abs(np.diff(opt, axis=0)).mean() if opt.shape[0] > 1 else 0.0
            gy = np.abs(np.diff(opt, axis=1)).mean() if opt.shape[1] > 1 else 0.0
            scores.append(gx + gy)
        return opts[int(np.argmax(scores))]

    def to2d(a, var=0, grid_shape=None):
        a = np.asarray(a)
        if a.ndim == 2 and a.shape[0] > 4 and a.shape[1] > 4:
            return a
        if a.ndim == 3 and a.shape[0] <= 4:
            return a[var]
        if a.ndim == 3 and a.shape[-1] <= 4:
            return a[..., var]
        if a.ndim == 2 and a.shape[1] <= 8:
            N, C = a.shape
            gs = grid_shape or infer_grid_shape(N)
            try:
                img3 = a.reshape(*gs, C)
                return img3[..., var]
            except Exception:
                vec = a[:, var]
                return best_reshape(vec, gs)
        if a.ndim == 3 and a.shape[-1] <= 8 and a.shape[0] > 1:
            member = a[0]
            return to2d(member, var=var, grid_shape=grid_shape)
        if a.ndim == 1:
            gs = grid_shape or infer_grid_shape(a.size)
            return best_reshape(a, gs)
        raise ValueError(f"Unhandled array shape in to2d: {a.shape}")

# helper: isotropic radial average of 2D power spectral density
def radial_psd(img2d, nbins=None):
    img = np.asarray(img2d, dtype=float)
    # remove mean (optional; keeps DC separate)
    img = img - img.mean()
    # 2D FFT and power
    F = fftpack.fftshift(fftpack.fft2(img))
    ps = np.abs(F) ** 2
    H, W = img.shape
    cy, cx = H // 2, W // 2
    Y, X = np.indices((H, W))
    R = np.sqrt((X - cx) ** 2 + (Y - cy) ** 2)
    rmax = int(np.floor(R.max()))
    if nbins is None:
        nbins = rmax
    # bin edges and centers
    bins = np.linspace(0.0, rmax, nbins + 1)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    inds = np.digitize(R.ravel(), bins) - 1
    ps_r = np.zeros(nbins, dtype=float)
    counts = np.zeros(nbins, dtype=int)
    flat_ps = ps.ravel()
    for i in range(nbins):
        mask = inds == i
        counts[i] = mask.sum()
        if counts[i] > 0:
            ps_r[i] = flat_ps[mask].mean()
        else:
            ps_r[i] = 0.0
    return bin_centers, ps_r, counts

# iterate models and compute ensemble-averaged spectra
models = list(data.keys())
specs = {}
for name in models:
    entry = data[name]
    # collect member images for chosen VAR_IDX
    member_imgs = []
    if entry.get("type") == "probabilistic":
        members = np.asarray(entry.get("ens_members"))
        # expected shapes: (n_members, N, C) or (n_members, N) etc.
        if members.ndim == 3:
            for m in range(members.shape[0]):
                img = to2d(members[m], VAR_IDX, GRID_SHAPE)
                member_imgs.append(img)
        elif members.ndim == 2:
            # treat as (n_members, N) or (N, C) -> single member
            img = to2d(members, VAR_IDX, GRID_SHAPE)
            member_imgs.append(img)
        else:
            # fallback: try first axis as members
            for m in range(members.shape[0]):
                img = to2d(members[m], VAR_IDX, GRID_SHAPE)
                member_imgs.append(img)
    else:
        # deterministic: use prediction as single member
        pred = entry.get("prediction")
        img = to2d(pred, VAR_IDX, GRID_SHAPE)
        member_imgs.append(img)

    if len(member_imgs) == 0:
        print(f"Warning: no members for model {name}; skipping")
        continue

    # compute spectra per member and average spectra
    bcs = None
    ps_accum = []
    for img in member_imgs:
        bc, ps_r, counts = radial_psd(img, nbins=min(GRID_SHAPE))
        if bcs is None:
            bcs = bc
        else:
            # ensure same binning (should be)
            if not np.allclose(bcs, bc):
                bc = bcs
        ps_accum.append(ps_r)
    ps_mean = np.mean(np.vstack(ps_accum), axis=0)
    specs[name] = (bcs, ps_mean)

# ground truth: find a target image (use first available)
target_entry = next((e for e in data.values() if "target" in e), None)
if target_entry is not None:
    tgt_img = to2d(target_entry["target"], VAR_IDX, GRID_SHAPE)
    tgt_bc, tgt_ps, _ = radial_psd(tgt_img, nbins=min(GRID_SHAPE))
else:
    tgt_img = None
    tgt_bc, tgt_ps = None, None

# # Plot ensemble-averaged spectra vs target
# plt.figure(figsize=(7, 5))
# for name, (k, ps) in specs.items():
#     # avoid DC (k=0) dominating; plot from k>=1
#     plt.loglog(k[1:], ps[1:], label=name, linewidth=1.5)
# if tgt_img is not None:
#     plt.loglog(tgt_bc[1:], tgt_ps[1:], 'k--', linewidth=2.0, label="Target")
# plt.xlabel("Wavenumber (radial bin)")
# plt.ylabel("Power spectral density (averaged)")
# var_name = "Precipitation" if VAR_IDX == 0 else "Temperature"
# plt.title(f"Ensemble-averaged energy spectra vs Target for {var_name}")
# plt.legend()
# plt.grid(True, which="both", ls=":", alpha=0.5)
# out_pdf = os.path.join(plots_dir, f"ensemble_vs_target_spectra_{VAR_IDX}.pdf")
# plt.savefig(out_pdf, bbox_inches="tight", dpi=200)
# print(f"Saved spectra plot to {out_pdf}")
# plt.show()

TITLE_FS = 20
LABEL_FS = 16
TICK_FS = 14
LEGEND_FS = 14

plt.rcParams.update({
    "axes.titlesize": TITLE_FS,
    "axes.labelsize": LABEL_FS,
    "xtick.labelsize": TICK_FS,
    "ytick.labelsize": TICK_FS,
    "legend.fontsize": LEGEND_FS,
})

fig, ax = plt.subplots(figsize=(6, 4))
for name, (k, ps) in specs.items():
    ax.loglog(k[1:], ps[1:], label=name, linewidth=1.5)
if tgt_img is not None:
    ax.loglog(tgt_bc[1:], tgt_ps[1:], 'k--', linewidth=2.0, label="Target")

var_name = "Precipitation" if VAR_IDX == 0 else "Temperature"
ax.set_xlabel("Wavenumber", fontsize=LABEL_FS)
ax.set_ylabel("Power", fontsize=LABEL_FS)
ax.set_title(f"Energy spectra for {var_name}", fontsize=TITLE_FS)
ax.legend(loc="best", fontsize=LEGEND_FS)
ax.grid(True, which="both", ls=":", alpha=0.5)
ax.tick_params(axis="both", which="major", labelsize=TICK_FS)

out_pdf = os.path.join(plots_dir, f"ensemble_vs_target_spectra_{VAR_IDX}.pdf")
fig.savefig(out_pdf, bbox_inches="tight", dpi=200)
print(f"Saved spectra plot to {out_pdf}")
plt.show()